# EvolveAI — Live Cognitive Runtime v1

A live, interactive research prototype combining the recurrent SNN, bounded Hugging Face streaming, curiosity-based learning control, lexical evidence retrieval, and an auditable belief ledger.

**This is not an LLM or AGI.** The SNN is byte-level next-byte prediction. The reasoning controller performs evidence retrieval and bookkeeping; it does not yet perform general semantic reasoning. The notebook is designed to expose its real behavior directly so we can improve it from observed failures.

In [ ]:
from pathlib import Path
import subprocess, sys, importlib
REPO = Path('/content/EvolveAI')
BRANCH = 'evolveai-maxgpu-one-shot'
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','https://github.com/swasthik-s/EvolveAI.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
%cd /content/EvolveAI/snn
%pip -q install 'datasets>=3.0,<5.0'
import torch
print('Branch:',BRANCH,'| CUDA:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))


## Start the live runtime

Mount Drive so model weights, episodic memory, semantic index and belief history survive a Colab reset. The first `/learn` initializes and trains the SNN; later learning cycles warm-start from its latest weights. Optimizer state is reinitialized each cycle.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from live_runtime import EvolveRuntime
WORKDIR = Path('/content/drive/MyDrive/EvolveAI/live_cognitive_v1')
AGENT = EvolveRuntime(
    workdir=WORKDIR,
    source_ids=['tinystories','wikitext','ag_news','gsm8k'],
    device='cuda' if torch.cuda.is_available() else 'cpu',
    block_size=128, batch_size=16, hidden_size=128, embedding_dim=64,
    learning_rate=0.0005, architecture='baseline', seed=43,
)
print('Runtime ready. Enter /help in the next cell.')


## Interact with EvolveAI

Run the cell and type commands. It remains active until `/quit`. Learning is explicit and bounded: `/learn 100` runs 100 online training steps, with a maximum of 2,000 per command. `/ask` retrieves evidence and separately displays the SNN's raw byte continuation so we can see what it actually produces.

In [ ]:
def show_help():
    print('''COMMANDS
/learn [steps]        Stream more HF data and update SNN weights (default 100)
/ask QUESTION         Retrieve evidence and show SNN byte continuation
/think QUESTION       Evidence retrieval only; displays source snippets
/generate PROMPT     Generate raw continuation from the SNN
/belief S|R|O|E|ID|P|W  Record reviewed evidence: subject|relation|object|evidence|source_id|polarity(+1/-1)|reliability(0..1)
/status               Show live system state
/help                 Show commands
/quit                 Save and exit
''')

show_help()
while True:
    try:
        command = input('\\nEVOLVEAI> ').strip()
    except (EOFError, KeyboardInterrupt):
        print('Stopping interactive session.')
        break
    if not command: continue
    if command == '/quit': break
    if command == '/help': show_help(); continue
    if command == '/status':
        import json; print(json.dumps(AGENT.status(),indent=2)); continue
    if command.startswith('/learn'):
        parts=command.split(); steps=int(parts[1]) if len(parts)>1 else 100
        AGENT.learn(steps=steps); continue
    if command.startswith('/think '):
        AGENT.think(command[7:]); continue
    if command.startswith('/generate '):
        prompt=command[10:]
        print('[SNN RAW CONTINUATION]\\n'+AGENT.generate(prompt)); continue
    if command.startswith('/ask '):
        question=command[5:]
        AGENT.think(question)
        print('[SNN RAW CONTINUATION — not a verified answer]\\n'+AGENT.generate(question, max_new_bytes=180)); continue
    if command.startswith('/belief '):
        fields=command[8:].split('|')
        if len(fields)!=7:
            print('Expected: subject|relation|object|evidence|source_id|polarity|reliability'); continue
        s,r,o,e,source,p,w=fields
        AGENT.observe_belief(s,r,o,e,source,polarity=int(p),reliability=float(w)); continue
    print('Unknown command. Type /help.')
AGENT.close()


## What to inspect during the live run

- Does the SNN's raw continuation become more coherent after successive learning cycles?
- Does retrieved evidence actually contain information relevant to your question?
- Does the belief ledger retain disagreement instead of silently overwriting it?
- Does the learning director change domain selection as losses evolve?

Try the same prompt before and after `/learn 200`. Keep the prompt fixed. Compare actual output, not only training loss. Also test questions unrelated to the streamed data: the correct behavior may be admitting that memory has no evidence.

**Known limitations:** retrieval is lexical BM25-style; the claim ledger requires explicit reviewed observations; it does not automatically infer claims from arbitrary prose. The SNN has no instruction tuning, tokenizer, attention mechanism, or natural-language reasoning objective. The first live milestone is an observable learning-and-memory loop, not general intelligence.